# KSI v2: Crash types and road design in the KSI model (draft)

Does adding past red-light-running, angle and rear-end crashes (#26) and LION road features (`preprocessing/roadway.py`) to the Q1 model improve future KSI prediction? (issue #32)

**Changes from Q1**
- **Sites are LION intersections** (#36), not 100 ft crash clusters: nodes where 3+ streets meet, with nodes of one divided-road intersection merged within 60 ft. A crash belongs to the nearest site within 100 ft.
- **Pooled 3-year test** (#31): fit once on prediction year 2022, score 2023–2025, and compare each site's KSI total over those three years.

**Crash types** (`CrashConfig`): *red-light running* is any vehicle with contributing factor "Traffic Control Disregarded" (a proxy that also covers stop signs); *rear-end* is one vehicle hit on its front and another on its back; *angle* is one hit on its front and another on its side.

In [11]:
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.append("..")
warnings.filterwarnings("ignore")

from model.count_models import HistoryRanker, NegativeBinomialModel
from preprocessing.config import SiteConfig
from preprocessing.panel import load_panel
from preprocessing.roadway import ROADWAY_FEATURES, add_roadway_features, load_roadway_features
from preprocessing.bicycle import add_bicycle_features, load_bicycle_features
from validation.validate import pooled_holdout

site_config = SiteConfig(source="lion")
panel, scoring, sites = load_panel(site_config=site_config)

panel = add_roadway_features(
    panel,
    load_roadway_features(sites, site_config=site_config)
)

panel = add_bicycle_features(
    panel,
    load_bicycle_features(
        sites,
        path="../data/bicycle/bicycle_counts.csv",
        site_config=site_config,
        radius_ft=1000,
    )
)






print(f"{panel['site_id'].nunique():,} intersection sites, prediction years {sorted(panel['year'].unique())}")
panel.groupby("year")[["ksi", "rlr", "rear_end", "angle", "crashes"]].sum()

165,674 intersection sites, prediction years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,ksi,rlr,rear_end,angle,crashes
year,,,,,
2022,2539,2860,22248,20109,92771
2023,2721,2996,21065,19656,88718
2024,2736,2640,19996,17673,83081
2025,2741,2324,19839,16703,80646


How many intersections actually have nearby counters?

In [12]:
print("Sites:", panel["site_id"].nunique())

print("\nBicycle coverage by year:")

display(
    panel.groupby("year").agg(
        sites=("site_id", "nunique"),
        sites_with_bicycle_data=(
            "bicycle_missing",
            lambda x: (x == 0).sum()
        ),
        sites_without_bicycle_data=(
            "bicycle_missing",
            lambda x: (x == 1).sum()
        ),
    )
)

Sites: 165674

Bicycle coverage by year:


,sites,sites_with_bicycle_data,sites_without_bicycle_data
year,,,
2022,165674,1555,164119
2023,165674,1826,163848
2024,165674,1937,163737
2025,165674,2067,163607


## 1. How common are red-light-running crashes?
Ko et al. (2012) found cameras pay off at intersections with 4+ red-light-running crashes a year. Few NYC intersections reach that by this proxy.

In [13]:
rlr_per_year = panel.query("year == 2025")["rlr_lag3"] / 3
pd.Series({
    "sites with any RLR crash (2022–24)": (rlr_per_year > 0).sum(),
    "1+ per year": (rlr_per_year >= 1).sum(),
    "2+ per year": (rlr_per_year >= 2).sum(),
    "4+ per year (Ko rule)": (rlr_per_year >= 4).sum(),
})

sites with any RLR crash (2022–24)    6229
1+ per year                            453
2+ per year                             30
4+ per year (Ko rule)                    3
dtype: int64

## 2. Model comparison (pooled 2023–2025)
Top-k capture: share of all 2023–25 KSI crashes at the top-k sites (higher is better). Deviance: fit across all sites (lower is better).

In [14]:
Q1 = [f"log1p_{c}_lag3" for c in ["pdo", "injury_nonsevere", "ksi_nonfatal", "fatal", "ped_injured", "cyc_injured"]]

BIKE = ["log1p_bike_volume", "bicycle_missing"]

TYPES = [f"log1p_{c}_lag3" for c in ["rlr", "rear_end", "angle"]]
# Every LION site is an intersection, so the leg indicators always sum to 1;
# drop legs_3 to make 3-leg intersections the reference.
ROAD = [f for f in ROADWAY_FEATURES if f != "legs_3"]
models = {
    "Past KSI (baseline)": lambda: HistoryRanker("ksi_lag3", 3),

    "Q1 model": lambda: NegativeBinomialModel(Q1),

    "+ crash types": lambda: NegativeBinomialModel(Q1 + TYPES),

    "+ LION road": lambda: NegativeBinomialModel(Q1 + ROAD),

    "+ both": lambda: NegativeBinomialModel(Q1 + TYPES + ROAD),

    "+ bicycle exposure": lambda: NegativeBinomialModel(
        Q1 + TYPES + ROAD + BIKE
    ),
}
rows = []
for name, make in models.items():
    metrics, _ = pooled_holdout(make, panel, (2023, 2024, 2025), (100, 1000))
    rows.append(metrics.assign(model=name))
res = pd.concat(rows)
res["column"] = [m if pd.isna(k) else f"{m}{int(k)}" for m, k in zip(res["metric"], res["k"])]
res.pivot(index="model", columns="column", values="value").loc[list(models)].round(4)

column,capture100,capture1000,deviance
model,,,
Past KSI (baseline),0.0198,0.1107,1.8325
Q1 model,0.0254,0.1478,0.2058
+ crash types,0.0248,0.1455,0.2031
+ LION road,0.0251,0.1509,0.2018
+ both,0.0253,0.1460,0.1995
+ bicycle exposure,0.0253,0.1458,0.1995


## 3. Rate ratios (full model, trained on 2022)
Crash counts: effect of doubling the past count, holding the others fixed. Road features: effect of one unit (a 0/1 flag, one lane, 10 ft of width, 1 ft of lane width).

In [15]:
train = panel[panel["year"] < 2023]
final = NegativeBinomialModel(Q1 + TYPES + ROAD + BIKE).fit(
    train, train["ksi"], groups=train["site_id"]
)

coef = final.coefficients().loc[TYPES + Q1 + ROAD + BIKE]
# rate ratios are per unit of log(1 + count); ratio ** log(2) is the effect of doubling the count
logged = coef.index.isin(TYPES + Q1)
coef.loc[logged, ["ratio", "ci_low", "ci_high"]] **= np.log(2)
coef.round(3)

,coef,ratio,ci_low,ci_high,p_value
log1p_rlr_lag3,-0.176,0.885,0.822,0.953,0.001
log1p_rear_end_lag3,-0.271,0.829,0.773,0.889,0.000
log1p_angle_lag3,-0.304,0.810,0.758,0.866,0.000
log1p_pdo_lag3,1.150,2.219,2.082,2.366,0.000
log1p_injury_nonsevere_lag3,0.747,1.678,1.544,1.824,0.000
log1p_ksi_nonfatal_lag3,0.075,1.053,0.970,1.143,0.216
log1p_fatal_lag3,-0.071,0.952,0.764,1.185,0.659
log1p_ped_injured_lag3,-0.107,0.928,0.873,0.988,0.019
log1p_cyc_injured_lag3,0.205,1.153,1.080,1.231,0.000
legs_4,0.364,1.440,1.287,1.611,0.000
